# Unity Catalog

A brief overview of the main Unity Catalog features, with syntax and small examples.

## 1. What Unity Catalog gives you

Unity Catalog is Databricks' governance layer for data, AI and other securables. It gives you:
* a three-level namespace: **catalog.schema.object**
* centralized permissions
* data lineage and auditability
* data discovery with comments and tags
* governance features such as PII classification, ABAC policies, row filters and column masks

---

## 2. Three-level namespace

| Level | Example | Purpose |
|---|---|---|
| Catalog | `training` | top-level container, often by domain or environment |
| Schema | `training.bronze` | logical grouping inside a catalog |
| Object | `training.bronze.customers` | table, view, volume, function, model, etc. |

### Syntax
```sql
CREATE CATALOG IF NOT EXISTS training;
CREATE SCHEMA IF NOT EXISTS training.bronze;
CREATE TABLE IF NOT EXISTS training.bronze.customers (
  customer_id BIGINT,
  customer_name STRING
);
```

---

## 3. Permissions and grants

Unity Catalog controls access with `GRANT` / `REVOKE`.

### Syntax
```sql
GRANT USE CATALOG ON CATALOG training TO `account users`;
GRANT USE SCHEMA ON SCHEMA training.bronze TO `account users`;
GRANT SELECT ON TABLE training.bronze.customers TO `one_developers`;
REVOKE SELECT ON TABLE training.bronze.customers FROM `one_developers`;
```

### Check permissions
```sql
SHOW GRANTS ON TABLE training.bronze.customers;
```



---

## 5. Views and functions

### View syntax
```sql
CREATE OR REPLACE VIEW training.silver.active_customers AS
SELECT *
FROM training.bronze.customers
WHERE customer_name IS NOT NULL;
```

### Function syntax
```sql
CREATE OR REPLACE FUNCTION training.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).*(@.*$)', '$1***$2');
```

Views help standardize logic. Functions help reuse logic, especially for governance and masking.

---

## 6. Comments for documentation

Comments improve discoverability in Catalog Explorer.

### Syntax
```sql
COMMENT ON TABLE training.bronze.customers IS 'Raw customer records from the source system';
COMMENT ON COLUMN training.bronze.customers.customer_name IS 'Customer full name';
```

---

## 7. Tags

Tags attach metadata to catalogs, schemas, tables and columns.

### Table tag syntax
```sql
SET TAG ON TABLE training.bronze.customers domain = customer;
SET TAG ON TABLE training.bronze.customers quality_tier = bronze;
```

### Column tag syntax
```sql
SET TAG ON COLUMN training.bronze.customers.customer_name pii = true;
SET TAG ON COLUMN training.bronze.customers.email pii_type = email;
```

### Query tags
```sql
SELECT catalog_name, schema_name, table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE tag_name = 'pii';
```

Tags are often the bridge between discovery and policy enforcement.

---

## 8. PII and data classification

Unity Catalog can automatically classify sensitive data such as PII, financial data and credentials, and store the results for governance workflows. Databricks data classification uses an agent to scan tables and can auto-tag sensitive columns, and the results are queryable in `system.data_classification.results`. [Data Classification](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/data-classification/) [Data classification system table reference](https://learn.microsoft.com/en-us/azure/databricks/admin/system-tables/data-classification/)

### Enable it first
`system.data_classification.results` does **not exist until Data Classification is enabled** on at least one catalog.
1. Catalog Explorer > open the catalog (e.g. `training`) > **Details** tab
2. Next to **Data Classification**, click **Enable**, choose the schemas, click **Save**
3. A background job scans the tables (new tables are usually scanned within 24 hours)

Requirements:
* catalog owner or `MANAGE` on the catalog
* serverless compute (the results table can only be queried from serverless)
* by default only **account admins** can read the results table; they can grant access to others

### Example query (after enabling)
```sql
SELECT *
FROM system.data_classification.results
WHERE catalog_name = 'training'
  AND confidence = 'HIGH';
```

### Fallback: tag PII manually and query the tags
If classification is not enabled, tag PII columns yourself and find them with `information_schema`:
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;

SELECT catalog_name, schema_name, table_name, column_name, tag_value AS pii_type
FROM system.information_schema.column_tags
WHERE tag_name = 'pii_type';
```

This helps you find columns that may need masking or row-level protection.

---

## 9. Row filters

A row filter limits which rows a user can see.

### Manual row filter on one table
```sql
ALTER TABLE training.silver.orders
SET ROW FILTER training.default.us_only_filter ON (country);
```

The function must return `BOOLEAN`: `TRUE` keeps the row.

### Example row filter function
```sql
CREATE OR REPLACE FUNCTION training.default.us_only_filter(country STRING)
RETURNS BOOLEAN
RETURN country = 'USA';
```

Use row filters when the same table should show different rows to different users.

---

## 10. Column masks

A column mask changes the value a user sees in a column.

### Manual mask on one table
```sql
ALTER TABLE training.silver.customers
ALTER COLUMN email
SET MASK training.default.mask_email;
```

### Example masking function
```sql
CREATE OR REPLACE FUNCTION training.default.mask_phone(phone STRING)
RETURNS STRING
RETURN concat('***-***-', right(phone, 4));
```

Column masks are useful for email, phone, SSN, salary and similar fields.

---

## 11. ABAC policies

ABAC means **attribute-based access control**. In Unity Catalog, ABAC policies use **governed tags** to dynamically apply row filters or column masks across many tables. Governed tags are the recommended tag type for ABAC policies. `CREATE POLICY` is the policy syntax; it is different from a manual `ALTER TABLE ... SET ROW FILTER` or `SET MASK` on one table. If enabled in your workspace, ABAC policies can be attached at metastore, catalog, schema or table scope and match tables/columns by governed tags. [Create and manage policies](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies) [Requirements, quotas, and limitations](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/requirements)

### Example governed-tag column mask policy for PII email
```sql
CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users` EXCEPT `balaji.raghu@one-consultants.com`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

### Example row filter policy
```sql
CREATE POLICY region_filter
ON SCHEMA training.silver
ROW FILTER training.default.us_only_filter
TO `one_developers`
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);
```

### Check policies
```sql
SHOW POLICIES ON SCHEMA training.silver;
SHOW EFFECTIVE POLICIES ON TABLE training.silver.customers;
```

**Difference from manual filters and masks:**
* manual row filters / masks apply to one table
* ABAC policies apply automatically to any table or column whose governed tags match the policy

### Step-by-step: ABAC row filter policy

**Goal:** members of `one_developers` only see customers from the `AMERICA` region in any **high-sensitivity** table of `training.silver`. Everyone else sees all rows.

**Principals used** (from this workspace)

| Principal | Type | Role in the example |
|---|---|---|
| `one_developers` | account group (11 members) | filtered by the policy (`TO`) |
| `balaji.raghu@one-consultants.com` | user (trainer) | exempt from the policy (`EXCEPT`) |
| `account users` | built-in account group | everyone in the account |

> `admins` and `users` are **workspace-local** groups and cannot be used in Unity Catalog grants or policies. Use account groups (or `account users`) instead.

**Prerequisites**
* Compute: serverless, or standard / dedicated compute on **Databricks Runtime 16.4+** (older runtimes are denied access to ABAC-secured tables)
* Permission to create governed tags (account admin, or `CREATE` / `MANAGE` on governed tags) and `ASSIGN` on the tags
* `MANAGE` on the schema `training.silver` (to create the policy) and `APPLY TAG` on the table
* `CREATE GOVERNED TAG` in SQL needs Databricks SQL or Runtime 18.1+ (or use the UI: **Catalog > Govern > Governed Tags**)

> Run the steps below **one cell at a time, in order**.

#### Step 1 — Create a sample table with a region column
Builds `training.silver.customer_abac` from `samples.tpch` and gives `one_developers` read access.

In [0]:
%sql
CREATE OR REPLACE TABLE training.silver.customer_abac AS
SELECT c.c_custkey, c.c_name, c.c_mktsegment, r.r_name AS region
FROM samples.tpch.customer c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey;

GRANT SELECT ON TABLE training.silver.customer_abac TO `one_developers`;

In [0]:
%sql
-- Expected: 5 regions, ~150,000 customers each
SELECT region, count(*) AS customers
FROM training.silver.customer_abac
GROUP BY region
ORDER BY region;

#### Step 2 — Create the governed tags (once per account)
Needs governed-tag create permission and Databricks SQL / Runtime 18.1+ (serverless works). If a tag already exists, skip this cell. Alternative: **Catalog > Govern > Governed Tags** in the UI.

In [0]:
%sql
CREATE GOVERNED TAG sensitivity
  DESCRIPTION 'Data sensitivity classification'
  VALUES ('low', 'medium', 'high');

-- key-only tag (no values)
CREATE GOVERNED TAG geo_region
  DESCRIPTION 'Marks the column that holds the customer region';

In [0]:
%sql
SHOW GOVERNED TAGS;

#### Step 3 — Tag the table and the region column
* table tag `sensitivity = high` → used by the policy's `WHEN`
* column tag `geo_region` → used by the policy's `MATCH COLUMNS`

In [0]:
%sql
SET TAG ON TABLE  training.silver.customer_abac sensitivity = high;
SET TAG ON COLUMN training.silver.customer_abac.region geo_region;

#### Step 4 — Check the tags
Expect 2 rows: one table tag and one column tag.
> Tag changes can take a few minutes to take effect.

In [0]:
%sql
SELECT 'table' AS level, table_name, NULL AS column_name, tag_name, tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'training' AND schema_name = 'silver' AND table_name = 'customer_abac'
UNION ALL
SELECT 'column', table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'training' AND schema_name = 'silver' AND table_name = 'customer_abac';

#### Step 5 — Create the row filter function
The function returns `TRUE` for rows to keep. Keep **who** it applies to out of the function — that is handled by the policy's `TO` / `EXCEPT`.

In [0]:
%sql
CREATE OR REPLACE FUNCTION training.default.filter_america(region STRING)
RETURNS BOOLEAN
RETURN region = 'AMERICA';

#### Step 6 — Create the ABAC row filter policy

| Clause | Meaning |
|---|---|
| `ON SCHEMA` | scope: every table in `training.silver` (inherited) |
| `ROW FILTER` | the boolean function to apply |
| `TO` / `EXCEPT` | who is filtered / who is exempt |
| `WHEN` | only tables with tag `sensitivity = high` |
| `MATCH COLUMNS ... AS region` | finds the column tagged `geo_region` |
| `USING COLUMNS (region)` | passes that column to the function |

In [0]:
%sql
CREATE POLICY america_only
ON SCHEMA training.silver
COMMENT 'one_developers only see AMERICA customers on high-sensitivity tables'
ROW FILTER training.default.filter_america
TO `one_developers`
EXCEPT `balaji.raghu@one-consultants.com`        -- exempt trusted users / ETL service principals
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')        -- which tables
MATCH COLUMNS has_tag('geo_region') AS region    -- which column to pass to the function
USING COLUMNS (region);

#### Step 7 — Verify the policy
Three checks: policies on the schema, policies that apply to the table (incl. inherited), and the full policy definition.

In [0]:
%sql
SHOW POLICIES ON SCHEMA training.silver;

In [0]:
%sql
SHOW EFFECTIVE POLICIES ON TABLE training.silver.customer_abac;

In [0]:
%sql
DESCRIBE POLICY america_only ON SCHEMA training.silver;

#### Step 8 — Test as different users
Run the next cell yourself, then ask a member of `one_developers` (e.g. Philippe Cooman, Sravan Koutharapu) to run it.

| Who runs the query | Result |
|---|---|
| member of `one_developers` | 1 row: `AMERICA` (~150,000) |
| `balaji.raghu@one-consultants.com` (EXCEPT) or anyone not in `one_developers` | all 5 regions (~750,000) |

In [0]:
%sql
SELECT region, count(*) AS customers
FROM training.silver.customer_abac
GROUP BY region
ORDER BY region;

#### Step 9 — Extend to more tables (no new policy needed)
Create a second table in `training.silver` and tag it the same way — the existing `america_only` policy applies to it automatically.

In [0]:
%sql
CREATE OR REPLACE TABLE training.silver.orders_by_region AS
SELECT r.r_name AS region,
       count(*) AS order_count,
       round(sum(o.o_totalprice), 2) AS total_price
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation n   ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r   ON n.n_regionkey = r.r_regionkey
GROUP BY r.r_name;

GRANT SELECT ON TABLE training.silver.orders_by_region TO `one_developers`;

SET TAG ON TABLE  training.silver.orders_by_region sensitivity = high;
SET TAG ON COLUMN training.silver.orders_by_region.region geo_region;

In [0]:
%sql
-- one_developers members see only the AMERICA row; you see all 5
SELECT * FROM training.silver.orders_by_region ORDER BY region;

#### Step 10 — Clean up
> Drop the policy **before** deleting a governed tag it uses — otherwise queries in the policy's scope fail.

In [0]:
%sql
DROP POLICY america_only ON SCHEMA training.silver;

UNSET TAG ON COLUMN training.silver.orders_by_region.region geo_region;
UNSET TAG ON TABLE  training.silver.orders_by_region sensitivity;
UNSET TAG ON COLUMN training.silver.customer_abac.region geo_region;
UNSET TAG ON TABLE  training.silver.customer_abac sensitivity;

DROP FUNCTION IF EXISTS training.default.filter_america;
DROP TABLE IF EXISTS training.silver.orders_by_region;
DROP TABLE IF EXISTS training.silver.customer_abac;

-- Optional: remove the governed tags (only if no other policy uses them)
-- DROP GOVERNED TAG geo_region;
-- DROP GOVERNED TAG sensitivity;


## 12. Auditing and lineage

Unity Catalog also supports lineage and audit-style analysis through system tables.

### Example lineage query
```sql
SELECT *
FROM system.access.table_lineage
WHERE target_table_full_name = 'training.silver.customers';
```

### Example access history query
```sql
SELECT *
FROM system.access.audit
WHERE service_name = 'unityCatalog';
```

---

## 13. Common governance pattern

A common pattern is:
1. store data in Unity Catalog tables
2. add comments and tags
3. classify sensitive columns
4. apply row filters or masks manually for one table
5. move to ABAC policies when you want the rule to apply dynamically across many tables

### Example sequence
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;
SET TAG ON TABLE training.silver.customers sensitivity = high;

CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

---

## 14. Quick summary

| Feature | Main syntax |
|---|---|
| Create schema | `CREATE SCHEMA training.bronze` |
| Grant access | `GRANT SELECT ON TABLE ...` |
| Comment | `COMMENT ON TABLE ...` |
| Tag | `SET TAG ON TABLE ...` |
| Query tags | `system.information_schema.*_tags` |
| Data classification | `system.data_classification.results` |
| Manual row filter | `ALTER TABLE ... SET ROW FILTER ...` |
| Manual mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK ...` |
| ABAC policy | `CREATE POLICY ... ROW FILTER / COLUMN MASK ...` |
| Show policies | `SHOW POLICIES ...` |

Use Unity Catalog when you want **data organization, access control, discovery and governance in one place**.